# Los datos de `medium`, hecho por hecho

**Pregunta.** Que dicen los datos del deposito `medium` de `data/v2` sobre su geometria, sus productos, sus
ordenes y batches, sus reposiciones y su demanda, y si la co-ocurrencia de SKUs en una orden excede la que
daria el azar.

**Para que.** Es la fuente de cada numero y de cada figura del capitulo de la tesis que describe el deposito.
Cada seccion deja un archivo `.dat` en `escritos/05_deposito/datos/` que dibuja la figura correspondiente.

**Contenido.**
1. El espacio: bays, pasillos, transversales, dock, distancias.
2. Productos, merchants y la asignacion inicial.
3. Ordenes y batches.
4. Reposiciones y mudanzas.
5. Corte entre aprendizaje y medicion.
6. Perfil de actividad.
7. Lo que se pide junto: co-ocurrencia observada contra un modelo nulo por permutacion.
8. Resumen de numeros.

In [1]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr

from abs_affinity_based_slotting.config import dataset_dir, DOCK, inches_to_meters
from abs_affinity_based_slotting.data import WarehouseDataLoader, split_picking_events
from abs_affinity_based_slotting.simulation import split_replenishments
from abs_affinity_based_slotting.warehouse.distances import distance_to_dock

OUT = Path("../../escritos/05_deposito/datos"); OUT.mkdir(parents=True, exist_ok=True)
R = {}   # numeros que se usan en el capitulo

def cargar(nombre):
    ds = WarehouseDataLoader(dataset_dir("v2", nombre)).load_all()
    man = json.load(open(dataset_dir("v2", nombre) / "manifest.json"))
    return ds, man

ds, man = cargar("medium")
co, di, st, pe, re_ = ds.coordinates, ds.distances, ds.initial_stock, ds.picking_events, ds.replenishment_events
print(man["preset"]); print({k: v for k, v in man.items() if k != "preset"})

{'num_aisles': 8, 'bays_per_side': 16, 'num_cross_aisles': 2, 'shelves_per_bay': 5, 'bins_per_shelf': 6, 'batch_size': 25, 'batches_per_day': 45, 'num_merchants': 50, 'num_pickers': 20, 'num_replenishers': 5, 'empty_location_pct': 0.1}
{'warehouse': 'medium', 'seed': 124, 'start_timestamp': '2026-01-01T00:00:00', 'end_timestamp': '2026-07-01T00:00:00', 'calendar_months': 6, 'num_batches': 8145, 'num_orders': 203625, 'rows': {'coordinates': 257, 'distances': 32896, 'initial_stock': 7680, 'picking_events': 712137, 'replenishment_events': 76966}}


## 1. El espacio

In [2]:
bays = co[co["bay_id"] != DOCK].copy()
bays["aisle"] = bays["aisle"].astype(int); bays["bay_number"] = bays["bay_number"].astype(int)
dock_xy = co.loc[co["bay_id"] == DOCK, ["x", "y"]].iloc[0].to_numpy(dtype=float)
d_dock = distance_to_dock(di)
bays["d_dock_in"] = bays["bay_id"].map(d_dock)

# numeracion de bays: donde esta el bay 1 de cada pasillo (frente o fondo)
y1 = bays[bays["bay_number"] == 1].groupby("aisle")["y"].first()
L = float(bays["y"].max() - bays["y"].min())
print("dock en", dock_xy, "| bays:", len(bays), "| pasillos:", bays["aisle"].nunique(), "| bays por lado:", bays["bay_number"].max() // 2)
print("y del bay 1 por pasillo (frente =", bays["y"].min(), ", fondo =", bays["y"].max(), "):", y1.to_dict())
print(f"largo util del pasillo L = {L:.0f} in = {inches_to_meters(L):.1f} m; x de los pasillos:", sorted(bays.groupby('aisle')['x'].mean().round(0).astype(int).tolist()))
print(f"distancia al dock: min {bays['d_dock_in'].min():.0f} in ({bays.loc[bays['d_dock_in'].idxmin(), 'bay_id']}), max {bays['d_dock_in'].max():.0f} in = {inches_to_meters(bays['d_dock_in'].max()):.1f} m ({bays.loc[bays['d_dock_in'].idxmax(), 'bay_id']})")

# bays enfrentados a distancia cero
dm = di.set_index(["bay_a", "bay_b"])["distance_in"]
par = bays.sort_values(["aisle", "bay_number"]).groupby("aisle").head(2)
print("distancia entre A01-01 y A01-02 (enfrentados):", float(dm.get(("A01-01", "A01-02"), np.nan)))

# transversales: las coordenadas y de los bays saltan mas que el paso normal donde hay un pasillo transversal
odd = bays[(bays["side"] == "odd") & (bays["aisle"] == 1)].sort_values("y")
ys = odd["y"].to_numpy(dtype=float); dy = np.diff(ys); paso = float(np.median(dy))
cruces = [float((ys[k] + ys[k + 1]) / 2) for k in np.flatnonzero(dy > paso + 1)]
print(f"paso entre bays: {paso:.0f} in | transversales interiores (y del centro): {cruces} | ancho: {sorted(set((dy[dy > paso + 1] - paso).round()))} in")
# y la distancia por el grafo los usa: entre dos bays de pasillos vecinos a la altura de un transversal
if cruces:
    yc = cruces[0]; cerca = bays.iloc[(bays["y"] - yc).abs().argsort()]
    b1 = cerca[cerca["aisle"] == 1].iloc[0]; b2 = cerca[cerca["aisle"] == 2].iloc[0]
    d_graf = dm.get((b1["bay_id"], b2["bay_id"]), dm.get((b2["bay_id"], b1["bay_id"])))
    print(f"distancia entre {b1['bay_id']} y {b2['bay_id']}, junto al transversal: {d_graf:.0f} in (por el frente seria {abs(float(b2['x'])-float(b1['x'])) + 2*(float(b1['y'])-168):.0f})")
# corredor frontal y trasero, inferidos de la distancia entre bays frontales de pasillos vecinos
a1 = bays[(bays["aisle"] == 1) & (bays["y"] == bays["y"].min())].iloc[0]
a2 = bays[(bays["aisle"] == 2) & (bays["y"] == bays["y"].min())].iloc[0]
d12 = dm.get((a1["bay_id"], a2["bay_id"]), dm.get((a2["bay_id"], a1["bay_id"])))
y_front = float(a1["y"]) - (d12 - abs(float(a2["x"]) - float(a1["x"]))) / 2
b1 = bays[(bays["aisle"] == 1) & (bays["y"] == bays["y"].max())].iloc[0]
b2 = bays[(bays["aisle"] == 2) & (bays["y"] == bays["y"].max())].iloc[0]
d12b = dm.get((b1["bay_id"], b2["bay_id"]), dm.get((b2["bay_id"], b1["bay_id"])))
y_back = float(b1["y"]) + (d12b - abs(float(b2["x"]) - float(b1["x"]))) / 2
print(f"corredor frontal en y = {y_front:.0f}, trasero en y = {y_back:.0f}")
R["geometria"] = dict(L_in=L, y_front=y_front, y_back=y_back, cruces=cruces, dock=dock_xy.tolist(),
                      d_dock_max_in=float(bays["d_dock_in"].max()), ubic_por_bay=int(len(st) / len(bays)))
bays.assign(x=bays["x"].astype(float), y=bays["y"].astype(float))[["bay_id", "aisle", "bay_number", "side", "x", "y", "d_dock_in"]] \
    .to_csv(OUT / "plano.dat", sep=" ", index=False, float_format="%.1f")
print("ubicaciones por bay:", R["geometria"]["ubic_por_bay"])

dock en [   0. -200.] | bays: 256 | pasillos: 8 | bays por lado: 16
y del bay 1 por pasillo (frente = 168.0 , fondo = 1848.0 ): {1: 168.0, 2: 1848.0, 3: 168.0, 4: 1848.0, 5: 168.0, 6: 1848.0, 7: 168.0, 8: 1848.0}
largo util del pasillo L = 1680 in = 42.7 m; x de los pasillos: [96, 288, 480, 672, 864, 1056, 1248, 1440]
distancia al dock: min 464 in (A01-01), max 3488 in = 88.6 m (A08-01)
distancia entre A01-01 y A01-02 (enfrentados): 0.0
paso entre bays: 96 in | transversales interiores (y del centro): [756.0, 1356.0] | ancho: [np.float64(120.0)] in
distancia entre A01-13 y A02-20, junto al transversal: 408 in (por el frente seria 1728)
corredor frontal en y = 60, trasero en y = 1956
ubicaciones por bay: 30


In [3]:
# geometria de los tres depositos y la variante, para la tabla de tamanos
filas = []
for nombre in ["small", "medium", "medium-alpha-1.3", "big"]:
    d, m = cargar(nombre)
    b = d.coordinates[d.coordinates["bay_id"] != DOCK]
    dd = distance_to_dock(d.distances)
    p = d.picking_events
    filas.append(dict(deposito=nombre, pasillos=int(b["aisle"].nunique()), bays_por_lado=int(b["bay_number"].max() // 2),
                      transversales=m["preset"]["num_cross_aisles"], ubicaciones=len(d.initial_stock),
                      SKU=int(d.initial_stock["sku"].nunique()), merchants=m["preset"]["num_merchants"],
                      pickers=m["preset"]["num_pickers"], repositores=m["preset"]["num_replenishers"],
                      ordenes=int(p["order_id"].nunique()), batches=int(p["batch_id"].nunique()),
                      batches_dia=m["preset"].get("batches_per_day"), lineas=len(p),
                      reposiciones=len(d.replenishment_events),
                      L_m=round(inches_to_meters(float(b["y"].max() - b["y"].min())), 1),
                      d_dock_max_m=round(inches_to_meters(float(b["bay_id"].map(dd).max())), 1),
                      vacias=int(len(d.initial_stock) - d.initial_stock["sku"].nunique()),
                      alpha=m["preset"].get("pareto_alpha")))
tam = pd.DataFrame(filas).set_index("deposito").T
print(tam.to_string()); tam.to_csv(OUT / "tamanos.csv")
R["tamanos"] = {k: {kk: (None if pd.isna(vv) else vv) for kk, vv in v.items()} for k, v in tam.to_dict().items()}

deposito       small  medium medium-alpha-1.3      big
pasillos           4       8                8       16
bays_por_lado     10      16               16       24
transversales      1       2                2        3
ubicaciones     1920    7680             7680    27648
SKU             1728    6912             6912    24883
merchants          4      50               50      100
pickers            5      20               20       60
repositores        2       5                5       15
ordenes        27150  203625           203625   452500
batches         1086    8145             8145    18100
batches_dia        6      45               45      100
lineas         95043  712137           711860  1583010
reposiciones    9417   76966            77016   171333
L_m             25.0    42.7             42.7     65.2
d_dock_max_m    51.4    88.6             88.6    150.2
vacias           192     768              768     2765
alpha           None    None             None     None


## 2. Productos, merchants y la asignacion inicial

In [4]:
ocupadas = st.dropna(subset=["sku"]) if st["sku"].isna().any() else st[st["sku"].astype(str) != ""]
n_sku = ocupadas["sku"].nunique()
print("ubicaciones:", len(st), "| con SKU:", len(ocupadas), "| vacias:", len(st) - len(ocupadas), f"({(len(st)-len(ocupadas))/len(st):.1%})")
print("SKU distintas:", n_sku, "| una ubicacion por SKU:", ocupadas["sku"].is_unique)
print("unidades iniciales por ubicacion: min", int(ocupadas["units"].min()), "mediana", int(ocupadas["units"].median()), "max", int(ocupadas["units"].max()))
sku_merch = ocupadas.groupby("merchant_account_id")["sku"].nunique()
ord_merch = pe.groupby("order_id")["merchant_account_id"].first().value_counts(normalize=True)
print("merchants:", sku_merch.size, "| SKU por merchant: min", int(sku_merch.min()), "max", int(sku_merch.max()),
      f"| fraccion de ordenes por merchant: min {ord_merch.min():.3f} max {ord_merch.max():.3f}")

# picks de los seis meses por bay, segun la ubicacion INICIAL de cada SKU
lineas_sku = pe.groupby("sku").size()
bay0 = ocupadas.set_index("sku")["bay_id"]
picks_bay = lineas_sku.groupby(bay0.reindex(lineas_sku.index)).sum().reindex(bays["bay_id"]).fillna(0)
pb = bays.set_index("bay_id").assign(picks=picks_bay)
rho = spearmanr(pb["picks"], pb["d_dock_in"]).statistic
por_pasillo = pb.groupby("aisle")["picks"].sum() / pb["picks"].sum()
print(f"picks por bay: min {int(pb['picks'].min())} max {int(pb['picks'].max())} CV {pb['picks'].std()/pb['picks'].mean():.2f}")
print(f"Spearman(picks por bay, distancia al dock) = {rho:.3f}")
print("fraccion de picks por pasillo:", por_pasillo.round(3).to_dict())
pb.reset_index()[["bay_id", "aisle", "x", "y", "d_dock_in", "picks"]].to_csv(OUT / "picks_bay.dat", sep=" ", index=False, float_format="%.1f")
R["productos"] = dict(sku=int(n_sku), vacias=int(len(st) - len(ocupadas)), units_min=int(ocupadas["units"].min()), units_max=int(ocupadas["units"].max()),
                      sku_merch_min=int(sku_merch.min()), sku_merch_max=int(sku_merch.max()), ord_merch_min=float(ord_merch.min()), ord_merch_max=float(ord_merch.max()),
                      spearman_picks_dock=float(rho), pasillo_min=float(por_pasillo.min()), pasillo_max=float(por_pasillo.max()), cv_picks_bay=float(pb['picks'].std()/pb['picks'].mean()))

ubicaciones: 7680 | con SKU: 6912 | vacias: 768 (10.0%)
SKU distintas: 6912 | una ubicacion por SKU: True
unidades iniciales por ubicacion: min 4 mediana 16 max 122


merchants: 50 | SKU por merchant: min 107 max 157 | fraccion de ordenes por merchant: min 0.019 max 0.021


picks por bay: min 794 max 10420 CV 0.51
Spearman(picks por bay, distancia al dock) = -0.012
fraccion de picks por pasillo: {1: 0.119, 2: 0.122, 3: 0.125, 4: 0.12, 5: 0.143, 6: 0.121, 7: 0.119, 8: 0.131}


In [5]:
# extracto real: un batch de test con dos ordenes de merchants distintos, seis lineas
split = split_picking_events(pe, test_size=0.2)
b0 = split.test["batch_id"].iloc[len(split.test) // 2]
g = pe[pe["batch_id"] == b0].sort_values("timestamp")
ordenes = g.groupby("order_id")["merchant_account_id"].first()
dos = ordenes.drop_duplicates().index[:2].tolist()
ex = g[g["order_id"].isin(dos)].sort_values("timestamp").head(6)[["batch_id", "order_id", "timestamp", "sku", "location_name", "merchant_account_id", "quantity"]]
print(ex.to_string(index=False)); ex.to_csv(OUT / "extracto.csv", index=False)

    batch_id        order_id                  timestamp       sku location_name merchant_account_id  quantity
BATCH-007332 ORDER-000183277 2026-06-12 21:52:25.933333 SKU-04088   A01-05-A-05             MER-047         1
BATCH-007332 ORDER-000183277 2026-06-12 21:57:49.533333 SKU-00689   A02-01-B-01             MER-047         2
BATCH-007332 ORDER-000183276 2026-06-12 22:01:54.533333 SKU-00007   A03-27-E-05             MER-015         3
BATCH-007332 ORDER-000183276 2026-06-12 22:10:01.933333 SKU-05357   A06-06-A-05             MER-015         1
BATCH-007332 ORDER-000183277 2026-06-12 22:15:07.733333 SKU-00233   A08-03-A-06             MER-047         3
BATCH-007332 ORDER-000183277 2026-06-12 22:17:21.933333 SKU-06000   A08-03-B-01             MER-047         3


## 3. Ordenes y batches

In [6]:
lpo = pe.groupby("order_id").size()
print("lineas por orden:", lpo.value_counts(normalize=True).sort_index().round(3).to_dict())
print("unidades por linea:", pe["quantity"].value_counts(normalize=True).sort_index().round(3).to_dict())
print("merchants por orden:", pe.groupby("order_id")["merchant_account_id"].nunique().value_counts().to_dict())
print("ordenes por batch:", pe.groupby("batch_id")["order_id"].nunique().value_counts().to_dict())
print("ordenes que repiten SKU:", int((pe.groupby(["order_id", "sku"]).size() > 1).sum()),
      "| lineas que repiten (batch, SKU):", int((pe.groupby(["batch_id", "sku"]).size() - 1).clip(lower=0).sum()))
spb = pe.groupby("batch_id")["sku"].nunique(); mpb = pe.groupby("batch_id")["merchant_account_id"].nunique()
bpb = pe.groupby("batch_id")["bay_name"].nunique(); lpb = pe.groupby("batch_id").size()
print(f"SKU distintas por batch: min {spb.min()} p5 {int(spb.quantile(.05))} mediana {int(spb.median())} p95 {int(spb.quantile(.95))} max {spb.max()} | lineas por batch: mediana {int(lpb.median())}")
print(f"merchants por batch: min {mpb.min()} mediana {int(mpb.median())} max {mpb.max()} | bays distintos por batch: mediana {int(bpb.median())}")
t0 = pe.groupby("batch_id")["timestamp"].min().sort_values(); t1 = pe.groupby("batch_id")["timestamp"].max()
gap = t0.diff().dt.total_seconds().dropna() / 60; dur = (t1 - t0).dt.total_seconds() / 60
print(f"intervalo entre batches: media {gap.mean():.2f} min, desvio {gap.std():.2f} | duracion de un batch: min {dur.min():.0f} mediana {dur.median():.0f} max {dur.max():.0f} min")
horas = pe["timestamp"].dt.hour.value_counts(normalize=True).sort_index()
dias = pe["timestamp"].dt.dayofweek.value_counts(normalize=True).sort_index()
print(f"fraccion de lineas por hora del dia: min {horas.min():.3f} max {horas.max():.3f} | por dia de la semana: min {dias.min():.3f} max {dias.max():.3f}")
print("pickers por batch:", pe.groupby("batch_id")["user_id"].nunique().value_counts().to_dict(), "| pickers distintos:", pe["user_id"].nunique(), "| repositores distintos:", re_["user_id"].nunique(),
      "| personas en comun:", len(set(pe["user_id"]) & set(re_["user_id"])))
R["ordenes"] = dict(lineas_por_orden=lpo.value_counts(normalize=True).sort_index().round(4).to_dict(), sku_batch_min=int(spb.min()), sku_batch_med=int(spb.median()), sku_batch_max=int(spb.max()),
                    sku_batch_p5=int(spb.quantile(.05)), sku_batch_p95=int(spb.quantile(.95)), merch_batch_min=int(mpb.min()), merch_batch_med=int(mpb.median()), merch_batch_max=int(mpb.max()),
                    gap_min=float(gap.mean()), gap_sd=float(gap.std()), dur_min=float(dur.min()), dur_med=float(dur.median()), dur_max=float(dur.max()),
                    hora_min=float(horas.min()), hora_max=float(horas.max()), lineas_rep_batch=int((pe.groupby(["batch_id", "sku"]).size() - 1).clip(lower=0).sum()))

lineas por orden: {2: 0.252, 3: 0.249, 4: 0.25, 5: 0.249}
unidades por linea: {np.uint16(1): 0.333, np.uint16(2): 0.332, np.uint16(3): 0.335}
merchants por orden: {1: 203625}


ordenes por batch: {25: 8145}


ordenes que repiten SKU: 0 | lineas que repiten (batch, SKU): 33486


SKU distintas por batch: min 63 p5 74 mediana 83 p95 93 max 106 | lineas por batch: mediana 87
merchants por batch: min 13 mediana 20 max 25 | bays distintos por batch: mediana 72
intervalo entre batches: media 32.00 min, desvio 0.15 | duracion de un batch: min 21 mediana 29 max 38 min
fraccion de lineas por hora del dia: min 0.041 max 0.043 | por dia de la semana: min 0.138 max 0.144


pickers por batch: {1: 8145} | pickers distintos: 20 | repositores distintos: 5 | personas en comun: 0


In [7]:
# orden de las lineas dentro del batch: siguen (pasillo, bay) ascendente?  cuantas inversiones hay y
# cuantas caen en una ubicacion que fue destino de una mudanza
coord_bay = co.set_index("bay_id")
seq = pe.sort_values(["batch_id", "timestamp"]).copy()
seq["aisle"] = seq["bay_name"].map(coord_bay["aisle"]).astype(int); seq["bn"] = seq["bay_name"].map(coord_bay["bay_number"]).astype(int)
key = seq["aisle"] * 1000 + seq["bn"]
prev_same = seq["batch_id"].eq(seq["batch_id"].shift())
inv = (key < key.shift()) & prev_same
destinos = set(re_.loc[re_["source_location_id"] != re_["target_location_id"], "target_location_id"].astype(int))
en_destino = seq.loc[inv, "location_id"].astype(int).isin(destinos).mean()
print(f"inversiones del orden (pasillo, bay) por batch: media {inv.sum()/pe['batch_id'].nunique():.2f}; {en_destino:.0%} de ellas en una ubicacion destino de mudanza")
# tiempo entre lineas consecutivas del mismo bay (lo que cuesta una linea sin caminar)
dt = seq["timestamp"].diff().dt.total_seconds()
mismo_bay = prev_same & seq["bay_name"].eq(seq["bay_name"].shift())
print(f"segundos entre lineas consecutivas del mismo bay: mediana {dt[mismo_bay].median():.1f}, p90 {dt[mismo_bay].quantile(.9):.1f}")
R["ordenes"].update(inversiones_batch=float(inv.sum()/pe['batch_id'].nunique()), inversiones_en_destino=float(en_destino), seg_linea=float(dt[mismo_bay].median()))

inversiones del orden (pasillo, bay) por batch: media 4.04; 96% de ellas en una ubicacion destino de mudanza
segundos entre lineas consecutivas del mismo bay: mediana 15.0, p90 15.0


In [8]:
# un batch representativo de test para la figura: el de SKU distintas mas cercano a la mediana
cand = spb[spb.index.isin(split.test["batch_id"])]
b_rep = (cand - cand.median()).abs().idxmin()
g = seq[seq["batch_id"] == b_rep].reset_index(drop=True)
fig = g.assign(seq=np.arange(1, len(g) + 1), x=g["bay_name"].map(coord_bay["x"]).astype(float), y=g["bay_name"].map(coord_bay["y"]).astype(float))[["seq", "bay_name", "aisle", "bn", "x", "y", "order_id", "merchant_account_id"]]
fig.to_csv(OUT / "batch.dat", sep=" ", index=False, float_format="%.1f")
print(b_rep, "| lineas:", len(g), "| SKU distintas:", g["sku"].nunique(), "| bays distintos:", g["bay_name"].nunique(), "| pasillos:", sorted(g["aisle"].unique().tolist()), "| merchants:", g["merchant_account_id"].nunique())
R["batch_figura"] = dict(id=b_rep, lineas=len(g), sku=int(g["sku"].nunique()), bays=int(g["bay_name"].nunique()), pasillos=int(g["aisle"].nunique()))

BATCH-006520 | lineas: 89 | SKU distintas: 84 | bays distintos: 76 | pasillos: [1, 2, 3, 4, 5, 6, 7, 8] | merchants: 20


## 4. Reposiciones y mudanzas

In [9]:
mud = re_["source_location_id"] != re_["target_location_id"]
dias_hist = (pe["timestamp"].max() - pe["timestamp"].min()).days
print(f"reposiciones: {len(re_)} | mudanzas: {int(mud.sum())} ({mud.mean():.1%}) | por dia: {len(re_)/dias_hist:.0f} reposiciones, {mud.sum()/dias_hist:.0f} mudanzas")
# cuanto se repone vs cuanto se pide, por SKU
rep_sku = re_.groupby("sku")["quantity"].sum(); lin_sku = pe.groupby("sku").size().reindex(rep_sku.index).fillna(0); un_sku = pe.groupby("sku")["quantity"].sum().reindex(rep_sku.index).fillna(0)
q_evento = re_.groupby("sku")["quantity"].mean()
print(f"por SKU, unidades repuestas en total contra unidades pedidas: Pearson {pearsonr(rep_sku, un_sku)[0]:.2f} (se repone lo que se pide)")
print(f"por SKU, unidades por reposicion contra lineas pedidas: Spearman {spearmanr(q_evento, lin_sku.reindex(q_evento.index)).statistic:.2f} (la cantidad por evento casi no crece con la demanda, asi que las SKU pedidas se reponen mas veces)")
rep_n = re_.groupby("sku").size(); mud_n = re_[mud].groupby("sku").size()
print(f"por SKU, cantidad de reposiciones contra lineas: Spearman {spearmanr(rep_n, lin_sku.reindex(rep_n.index).fillna(0)).statistic:.2f} | unidades por reposicion: mediana {q_evento.median():.0f}, min {re_['quantity'].min()}, max {re_['quantity'].max()}")
top = lineas_sku.sort_values(ascending=False); top10 = top.index[: int(0.1 * len(top))]
print(f"el 10% de SKU mas pedidas hace el {top.loc[top10].sum()/top.sum():.0%} de las lineas y el {mud_n.reindex(top10).fillna(0).sum()/mud_n.sum():.0%} de las mudanzas")
# cuanto antes del pick llega la reposicion
nxt = pe[["timestamp", "location_id", "sku"]].sort_values("timestamp")
rr = re_[["timestamp", "target_location_id", "sku"]].rename(columns={"target_location_id": "location_id"}).sort_values("timestamp")
m = pd.merge_asof(rr, nxt.rename(columns={"timestamp": "t_pick"}), left_on="timestamp", right_on="t_pick", by=["location_id", "sku"], direction="forward")
lead = (m["t_pick"] - m["timestamp"]).dt.total_seconds()
print(f"segundos entre la reposicion y el siguiente pick de esa SKU en esa ubicacion: mediana {lead.median():.0f}, p90 {lead.quantile(.9):.0f}, sin pick posterior: {lead.isna().mean():.1%}")
# fraccion de SKU que nunca se mudaron, dia a dia
primera_mud = re_[mud].groupby("sku")["timestamp"].min()
fechas = pd.date_range(pe["timestamp"].min().normalize(), pe["timestamp"].max().normalize(), freq="D")
nunca = pd.Series([(primera_mud > f).sum() + (n_sku - len(primera_mud)) for f in fechas], index=fechas) / n_sku
print("fraccion de SKU nunca mudadas al 1 de feb / abr / jul:", [round(float(nunca.asof(pd.Timestamp(d))), 3) for d in ["2026-02-01", "2026-04-01", "2026-06-30"]])
pd.DataFrame({"dia": np.arange(len(fechas)), "fecha": fechas.strftime("%Y-%m-%d"), "fraccion": nunca.round(4).to_numpy()}).to_csv(OUT / "nunca_mudadas.dat", sep=" ", index=False)
R["reposiciones"] = dict(total=len(re_), mudanzas=int(mud.sum()), frac_mud=float(mud.mean()), por_dia=float(len(re_)/dias_hist), mud_por_dia=float(mud.sum()/dias_hist),
                         corr_unidades=float(pearsonr(rep_sku, un_sku)[0]), corr_qty_evento=float(spearmanr(q_evento, lin_sku.reindex(q_evento.index)).statistic), top10_lineas=float(top.loc[top10].sum()/top.sum()), top10_mud=float(mud_n.reindex(top10).fillna(0).sum()/mud_n.sum()),
                         lead_med_s=float(lead.median()), nunca_feb=float(nunca.asof(pd.Timestamp("2026-02-01"))), nunca_abr=float(nunca.asof(pd.Timestamp("2026-04-01"))), nunca_jun=float(nunca.asof(pd.Timestamp("2026-06-30"))))

reposiciones: 76966 | mudanzas: 34558 (44.9%) | por dia: 428 reposiciones, 192 mudanzas
por SKU, unidades repuestas en total contra unidades pedidas: Pearson 1.00 (se repone lo que se pide)
por SKU, unidades por reposicion contra lineas pedidas: Spearman 0.19 (la cantidad por evento casi no crece con la demanda, asi que las SKU pedidas se reponen mas veces)
por SKU, cantidad de reposiciones contra lineas: Spearman 0.70 | unidades por reposicion: mediana 19, min 6, max 123
el 10% de SKU mas pedidas hace el 64% de las lineas y el 66% de las mudanzas


segundos entre la reposicion y el siguiente pick de esa SKU en esa ubicacion: mediana 75, p90 75, sin pick posterior: 0.0%
fraccion de SKU nunca mudadas al 1 de feb / abr / jul: [0.757, 0.508, 0.318]


## 5. Corte entre aprendizaje y medicion

In [10]:
filas = []
for nombre in ["medium", "medium-alpha-1.3", "big"]:
    d, m = cargar(nombre)
    s = split_picking_events(d.picking_events, test_size=0.2)
    rt, rs = split_replenishments(d.replenishment_events, s.train, d.initial_stock, s.cutoff)
    antes = (d.replenishment_events["timestamp"] < s.cutoff).sum()
    tail = s.train[s.train["timestamp"] >= s.cutoff]
    filas.append(dict(deposito=nombre, corte=str(s.cutoff), batches_train=s.train["batch_id"].nunique(), batches_test=s.test["batch_id"].nunique(),
                      ordenes_train=s.train["order_id"].nunique(), ordenes_test=s.test["order_id"].nunique(), lineas_train=len(s.train), lineas_test=len(s.test),
                      dias_train=(s.cutoff - d.picking_events["timestamp"].min()).days, dias_test=(d.picking_events["timestamp"].max() - s.cutoff).days,
                      repos_movidas_a_train=int(len(rt) - antes), lineas_train_despues_del_corte=len(tail)))
corte = pd.DataFrame(filas).set_index("deposito").T; print(corte.to_string()); corte.to_csv(OUT / "corte.csv")
R["corte"] = corte.to_dict()

deposito                                            medium            medium-alpha-1.3                         big
corte                           2026-05-25 19:12:22.733333  2026-05-25 19:12:22.733333  2026-05-25 19:13:15.933333
batches_train                                         6516                        6516                       14480
batches_test                                          1629                        1629                        3620
ordenes_train                                       162900                      162900                      362000
ordenes_test                                         40725                       40725                       90500
lineas_train                                        569482                      569320                     1265889
lineas_test                                         142655                      142540                      317121
dias_train                                             144                      

## 6. Perfil de actividad (sobre el tramo de aprendizaje de `medium`)

In [11]:
tr = split.train
mes = pe.groupby(pe["timestamp"].dt.to_period("M")).size(); sem = pe.groupby(pe["timestamp"].dt.to_period("W")).size()
print("lineas por mes:", mes.to_dict()); print(f"lineas por semana completa: mediana {int(sem.iloc[1:-1].median())}, min {int(sem.iloc[1:-1].min())}, max {int(sem.iloc[1:-1].max())} (primera y ultima parciales: {int(sem.iloc[0])}, {int(sem.iloc[-1])})")
f = tr.groupby("sku").size().sort_values(ascending=False); acum = f.cumsum() / f.sum()
n50 = int((acum < 0.5).sum() + 1); n80 = int((acum < 0.8).sum() + 1)
x = np.linspace(0, 1, 101); y = np.interp(x, np.arange(1, len(f) + 1) / len(f), acum.to_numpy()); y[0] = 0
pd.DataFrame({"x": x, "y": y}).to_csv(OUT / "pareto.dat", sep=" ", index=False, float_format="%.4f")
print(f"SKU con picks en train: {len(f)} de {n_sku} | el 20% mas pedido explica {y[20]:.1%} | {n50} SKU explican el 50% ({n50/len(f):.1%}) y {n80} el 80% ({n80/len(f):.1%})")
# por merchant: que fraccion de sus SKU hace el 80% de sus picks
fm = tr.groupby(["merchant_account_id", "sku"]).size()
def frac80(s):
    s = s.sort_values(ascending=False); c = s.cumsum() / s.sum(); return ((c < 0.8).sum() + 1) / len(s)
f80 = fm.groupby(level=0).apply(frac80)
print(f"por merchant, fraccion de sus SKU que hace el 80% de sus picks: min {f80.min():.1%} max {f80.max():.1%}")
# estabilidad del ranking entre meses (periodo completo)
pm = pe.groupby([pe["timestamp"].dt.to_period("M"), "sku"]).size().unstack(fill_value=0)
tops = [set(pm.loc[p].sort_values(ascending=False).index[:50]) for p in pm.index]
solap = [len(tops[k] & tops[k + 1]) for k in range(len(tops) - 1)]
rhos = [spearmanr(pm.iloc[k], pm.iloc[k + 1]).statistic for k in range(len(pm) - 1)]
print("top 50 compartido entre meses consecutivos:", solap, "| Spearman entre meses consecutivos (todas las SKU):", [round(r, 2) for r in rhos])
# la variante concentrada
d_a, _ = cargar("medium-alpha-1.3"); fa = split_picking_events(d_a.picking_events, test_size=0.2).train.groupby("sku").size().sort_values(ascending=False); ca = fa.cumsum() / fa.sum()
print(f"medium-alpha-1.3: {int((ca < .5).sum() + 1)} SKU explican el 50% y el 20% mas pedido explica {np.interp(0.2, np.arange(1, len(fa)+1)/len(fa), ca.to_numpy()):.1%}")
R["perfil"] = dict(sku_train=len(f), top20=float(y[20]), n50=n50, n80=n80, f80_min=float(f80.min()), f80_max=float(f80.max()), top50_solap=solap, rho_meses=[float(r) for r in rhos],
                   sem_med=int(sem.iloc[1:-1].median()), alpha_n50=int((ca < .5).sum() + 1), alpha_top20=float(np.interp(0.2, np.arange(1, len(fa)+1)/len(fa), ca.to_numpy())))

lineas por mes: {Period('2026-01', 'M'): 121805, Period('2026-02', 'M'): 110117, Period('2026-03', 'M'): 121847, Period('2026-04', 'M'): 118205, Period('2026-05', 'M'): 121804, Period('2026-06', 'M'): 118359}
lineas por semana completa: mediana 27539, min 27328, max 27669 (primera y ultima parciales: 15699, 7886)
SKU con picks en train: 6912 de 6912 | el 20% mas pedido explica 76.3% | 318 SKU explican el 50% (4.6%) y 1733 el 80% (25.1%)
por merchant, fraccion de sus SKU que hace el 80% de sus picks: min 23.5% max 29.0%


top 50 compartido entre meses consecutivos: [50, 50, 50, 50, 50] | Spearman entre meses consecutivos (todas las SKU): [np.float64(0.77), np.float64(0.77), np.float64(0.77), np.float64(0.76), np.float64(0.76)]


medium-alpha-1.3: 227 SKU explican el 50% y el 20% mas pedido explica 81.8%


## 7. Lo que se pide junto

Para cada par de SKUs $(i,j)$ del mismo merchant, $n_{ij}$ es la cantidad de ordenes que traen a las dos.
El modelo nulo conserva, dentro de cada merchant, la cantidad de ordenes de cada tamano y la cantidad de
lineas de cada SKU, y rompe solo la relacion entre SKUs: se barajan las lineas entre las ordenes del
merchant (sin repetir una SKU dentro de una orden) y se vuelve a contar. $\hat n_{ij}$ es el promedio
sobre $K$ barajados, y el *lift* observado es $n_{ij} / \hat n_{ij}$. Se reporta sobre los pares con
$\hat n_{ij} \ge 20$, donde el cociente es estable, y tambien con $\hat n_{ij} \ge 5$. Para saber cuanto
se dispersa el lift por puro azar, un barajado mas se trata como si fuera lo observado y se le calcula la
misma banda.

In [12]:
def pares(order_idx, sku_idx, n):
    # pares (i<j) dentro de cada orden -> codigo i*n+j
    df = pd.DataFrame({"o": order_idx, "s": sku_idx}).sort_values(["o", "s"])
    m = df.merge(df, on="o"); m = m[m["s_x"] < m["s_y"]]
    return (m["s_x"].to_numpy(dtype=np.int64) * n + m["s_y"].to_numpy(dtype=np.int64))

def barajar(order_idx, sku_idx, sizes, rng):
    # permuta las SKU entre las ordenes (del mismo merchant) conservando el tamano de cada orden; repara repetidas
    s = sku_idx.copy(); rng.shuffle(s)
    for _ in range(50):
        df = pd.DataFrame({"o": order_idx, "s": s})
        dup = df.duplicated(["o", "s"]).to_numpy()
        if not dup.any(): break
        idx = np.flatnonzero(dup); otros = rng.integers(0, len(s), size=len(idx))
        s[idx], s[otros] = s[otros], s[idx]
    return s

t = time.time(); rng = np.random.default_rng(0); K = 20
skus = pd.Index(tr["sku"].unique()); n = len(skus)
obs_all, exp_all, rep_all = {}, {}, {}
for merch, g in tr.groupby("merchant_account_id"):
    oid = pd.factorize(g["order_id"])[0]; sid = skus.get_indexer(g["sku"]); sizes = np.bincount(oid)
    obs = pd.Series(pares(oid, sid, n)).value_counts()
    acc = {}
    for k in range(K):
        for code_, c in pd.Series(pares(oid, barajar(oid, sid, sizes, rng), n)).value_counts().items():
            acc[code_] = acc.get(code_, 0) + c
    exp = pd.Series(acc) / K
    # una replica mas, tratada como si fuera lo observado: el lift que da el puro azar
    rep_all[merch] = pd.Series(pares(oid, barajar(oid, sid, sizes, rng), n)).value_counts()
    obs_all[merch] = obs; exp_all[merch] = exp
obs = pd.concat(obs_all.values()); exp = pd.concat(exp_all.values()); rep1 = pd.concat(rep_all.values())
tabla = pd.DataFrame({"obs": obs, "exp": exp, "rep": rep1}).fillna(0)
print(f"{len(tabla)} pares con alguna coincidencia observada o esperada, en {time.time()-t:.0f} s")
for umbral in (20, 5):
    sel = tabla[tabla["exp"] >= umbral]; lift = sel["obs"] / sel["exp"]; lift0 = sel["rep"] / sel["exp"]
    print(f"exp >= {umbral}: {len(sel)} pares | lift observado: p5 {lift.quantile(.05):.2f} mediana {lift.median():.2f} p99 {lift.quantile(.99):.2f} max {lift.max():.2f}, pares > 1,5: {(lift > 1.5).sum()}"
          f" | lift de una replica al azar: p5 {lift0.quantile(.05):.2f} p99 {lift0.quantile(.99):.2f} max {lift0.max():.2f}, pares > 1,5: {(lift0 > 1.5).sum()}")
    R[f"lift_{umbral}"] = dict(pares=len(sel), p5=float(lift.quantile(.05)), med=float(lift.median()), p99=float(lift.quantile(.99)), max=float(lift.max()), mayores_15=int((lift > 1.5).sum()),
                               nulo_p5=float(lift0.quantile(.05)), nulo_p99=float(lift0.quantile(.99)), nulo_max=float(lift0.max()), nulo_mayores_15=int((lift0 > 1.5).sum()))
sel = tabla[tabla["exp"] >= 5].copy(); sel["lift"] = sel["obs"] / sel["exp"]
muestra = sel.sample(min(4000, len(sel)), random_state=0)
muestra[["obs", "exp", "rep"]].to_csv(OUT / "lift.dat", sep=" ", index=False, float_format="%.2f")

423079 pares con alguna coincidencia observada o esperada, en 29 s
exp >= 20: 6179 pares | lift observado: p5 0.79 mediana 1.00 p99 1.34 max 1.54, pares > 1,5: 1 | lift de una replica al azar: p5 0.78 p99 1.33 max 1.62, pares > 1,5: 8
exp >= 5: 25391 pares | lift observado: p5 0.55 mediana 0.99 p99 1.75 max 2.63, pares > 1,5: 975 | lift de una replica al azar: p5 0.55 p99 1.78 max 2.75, pares > 1,5: 1066


## 8. Resumen de numeros

In [13]:
json.dump(R, open(OUT / "resumen.json", "w"), indent=1, default=str)
print(json.dumps(R, indent=1, default=str)[:6000])

{
 "geometria": {
  "L_in": 1680.0,
  "y_front": "60.0",
  "y_back": "1956.0",
  "cruces": [
   756.0,
   1356.0
  ],
  "dock": [
   0.0,
   -200.0
  ],
  "d_dock_max_in": 3488.0,
  "ubic_por_bay": 30
 },
 "tamanos": {
  "small": {
   "pasillos": 4,
   "bays_por_lado": 10,
   "transversales": 1,
   "ubicaciones": 1920,
   "SKU": 1728,
   "merchants": 4,
   "pickers": 5,
   "repositores": 2,
   "ordenes": 27150,
   "batches": 1086,
   "batches_dia": 6,
   "lineas": 95043,
   "reposiciones": 9417,
   "L_m": 25.0,
   "d_dock_max_m": 51.4,
   "vacias": 192,
   "alpha": null
  },
  "medium": {
   "pasillos": 8,
   "bays_por_lado": 16,
   "transversales": 2,
   "ubicaciones": 7680,
   "SKU": 6912,
   "merchants": 50,
   "pickers": 20,
   "repositores": 5,
   "ordenes": 203625,
   "batches": 8145,
   "batches_dia": 45,
   "lineas": 712137,
   "reposiciones": 76966,
   "L_m": 42.7,
   "d_dock_max_m": 88.6,
   "vacias": 768,
   "alpha": null
  },
  "medium-alpha-1.3": {
   "pasillos": 8,
   "ba